# Pipeline for HPE with mediapipe

03/10/2026 | 
Author: Cristian Ortega Singer | 
Mail: cris.ortega@fau.de


ChatAI by GWDG Gesellschaft für wissenschaftliche Datenverarbeitung was used for asistance in developing parts of the pipeline and for debbuging:

>Doosthosseini, Ali, Jonathan Decker, Hendrik Nolte, and Julian Kunkel. 2026. “SAIA: A Seamless Slurm-Native Solution for HPC-Based Services.” The Journal of Supercomputing 82 (7): 403. https://doi.org/10.1007/s11227-026-08508-3.

Setup Environment

In [1]:
%pip install jupyterlab ipykernel opencv-python pandas pyarrow tqdm numpy matplotlib mediapipe
!python -m ipykernel install --user --name slam-pose --display-name "Python (slam-pose)"

Note: you may need to restart the kernel to use updated packages.
Installed kernelspec slam-pose in C:\Users\CrisO\AppData\Roaming\jupyter\kernels\slam-pose


Load dependencies

In [2]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Load MediaPipe model

In [3]:
MODEL_PATH = "../models/mediapipe/pose_landmarker_heavy.task"

def create_pose_landmarker():
    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    return vision.PoseLandmarker.create_from_options(options)


Function for Pose Extraction

In [ ]:
def extract_pose_from_video(video_path, sample_every_n=3):
    rows = []

    with create_pose_landmarker() as pose_landmarker:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)

        if fps <= 0 or np.isnan(fps):
            fps = 25.0

        frame_idx = 0
        last_timestamp_ms = -1

        while True:
            ok, frame = cap.read()
            if not ok:
                break

            if frame_idx % sample_every_n != 0:
                frame_idx += 1
                continue

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            timestamp_ms = int((frame_idx / fps) * 1000)

            # make absolutely sure timestamps are strictly increasing
            if timestamp_ms <= last_timestamp_ms:
                timestamp_ms = last_timestamp_ms + 1

            result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)
            last_timestamp_ms = timestamp_ms

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                world_landmarks = (
                    result.pose_world_landmarks[0]
                    if result.pose_world_landmarks
                    else None
                )

                for i, lm in enumerate(landmarks):
                    row = {
                        "video": Path(video_path).name,
                        "frame_idx": frame_idx,
                        "timestamp_ms": timestamp_ms,
                        "landmark_id": i,
                        "x": lm.x,
                        "y": lm.y,
                        "z": lm.z,
                        "visibility": getattr(lm, "visibility", np.nan),
                        "presence": getattr(lm, "presence", np.nan),
                    }

                    if world_landmarks:
                        wlm = world_landmarks[i]
                        row.update({
                            "world_x": wlm.x,
                            "world_y": wlm.y,
                            "world_z": wlm.z,
                        })
                    else:
                        row.update({
                            "world_x": np.nan,
                            "world_y": np.nan,
                            "world_z": np.nan,
                        })

                    rows.append(row)

            frame_idx += 1

        cap.release()

    return pd.DataFrame(rows)

Batch Processing

In [ ]:
VIDEO_DIR = Path("../data/source_videos")
OUT_DIR = Path("../data/derived/mediapipe_pose")
OUT_DIR.mkdir(parents=True, exist_ok=True)

video_files = sorted([p for p in VIDEO_DIR.iterdir() if p.suffix.lower() == ".mp4"])

print("Found videos:", len(video_files))

for video_path in tqdm(video_files):
    print("Processing:", video_path.name)

    out_file = OUT_DIR / f"{video_path.stem}.parquet"
    if out_file.exists():
        continue

    df = extract_pose_from_video(video_path, sample_every_n=8)
    df.to_parquet(out_file, index=False)

Found videos: 4


  0%|          | 0/4 [00:00<?, ?it/s]

Processing: 2013-04-26_bzkLtFLpWEg.mp4
Processing: 2015-02-16_VGa4roB2tRE.mp4


 50%|█████     | 2/4 [03:27<03:27, 103.69s/it]

Processing: 2015-02-23_kLLyjW96Ez0.mp4


 75%|███████▌  | 3/4 [06:50<02:25, 145.13s/it]

Processing: 2017-06-19_0XpMVQgZwwU.mp4


100%|██████████| 4/4 [11:08<00:00, 167.19s/it]
